In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/edstats_tratado.csv")
print("Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/edstats_tratado.csv'

In [2]:
import os
print("Onde estou agora:", os.getcwd())

Onde estou agora: C:\Users\User\acelera-ai-educacao


In [3]:
import pandas as pd

df = pd.read_csv("data/processed/edstats_tratado.csv")
print("Shape:", df.shape)
df.head()

Shape: (35952, 6)


,Country Name,Country Code,Indicator Name,Indicator Code,Ano,Valor
0,Aruba,ABW,"Primary completion rate, both sexes (%)",SE.PRM.CMPT.ZS,1990,101.619118
1,Aruba,ABW,"Primary completion rate, both sexes (%)",SE.PRM.CMPT.ZS,1991,101.619118
2,Aruba,ABW,"Primary completion rate, both sexes (%)",SE.PRM.CMPT.ZS,1992,101.619118
3,Aruba,ABW,"Primary completion rate, both sexes (%)",SE.PRM.CMPT.ZS,1993,101.619118
4,Aruba,ABW,"Primary completion rate, both sexes (%)",SE.PRM.CMPT.ZS,1994,101.619118


In [4]:
# para cada país + indicador, pega o primeiro e o último ano com valor válido
def calcular_crescimento(grupo):
    grupo_valido = grupo.dropna(subset=["Valor"])
    if grupo_valido.empty:
        return pd.Series({"Valor_Inicial": None, "Ano_Inicial": None,
                           "Valor_Final": None, "Ano_Final": None,
                           "Crescimento_Absoluto": None, "Crescimento_Percentual": None})
    
    primeiro = grupo_valido.iloc[0]
    ultimo = grupo_valido.iloc[-1]
    
    crescimento_absoluto = ultimo["Valor"] - primeiro["Valor"]
    if primeiro["Valor"] != 0:
        crescimento_percentual = (crescimento_absoluto / primeiro["Valor"]) * 100
    else:
        crescimento_percentual = None
    
    return pd.Series({
        "Valor_Inicial": primeiro["Valor"], "Ano_Inicial": primeiro["Ano"],
        "Valor_Final": ultimo["Valor"], "Ano_Final": ultimo["Ano"],
        "Crescimento_Absoluto": crescimento_absoluto,
        "Crescimento_Percentual": crescimento_percentual
    })

df_ordenado = df.sort_values(["Country Code", "Indicator Code", "Ano"])

df_crescimento = df_ordenado.groupby(
    ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
).apply(calcular_crescimento, include_groups=False).reset_index()

print("Shape:", df_crescimento.shape)
df_crescimento.head(10)

Shape: (1284, 10)


,Country Name,Country Code,Indicator Name,Indicator Code,Valor_Inicial,Ano_Inicial,Valor_Final,Ano_Final,Crescimento_Absoluto,Crescimento_Percentual
0,Afghanistan,AFG,Expenditure on primary as % of government expe...,SE.XPD.PRIM.ZS,62.136471,1990.0,56.695492,2017.0,-5.440979,-8.756498
1,Afghanistan,AFG,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,3.461490,1990.0,3.317540,2017.0,-0.143950,-4.158613
2,Afghanistan,AFG,"Gross enrolment ratio, primary, both sexes (%)",SE.PRM.ENRR,30.091150,1990.0,111.877083,2017.0,81.785933,271.793972
3,Afghanistan,AFG,"Gross enrolment ratio, secondary, both sexes (%)",SE.SEC.ENRR,11.222830,1990.0,55.644409,2017.0,44.421579,395.814425
4,Afghanistan,AFG,"Primary completion rate, both sexes (%)",SE.PRM.CMPT.ZS,29.553841,1990.0,29.553841,2017.0,0.000000,0.000000
5,Afghanistan,AFG,"Teachers in primary education, both sexes (num...",SE.PRM.TCHR,15106.000000,1990.0,142880.000000,2017.0,127774.000000,845.849331
6,Albania,ALB,Expenditure on primary as % of government expe...,SE.XPD.PRIM.ZS,54.800331,1990.0,54.800331,2017.0,0.000000,0.000000
7,Albania,ALB,Government expenditure on education as % of GD...,SE.XPD.TOTL.GD.ZS,3.263300,1990.0,3.539440,2017.0,0.276140,8.461986
8,Albania,ALB,"Gross enrolment ratio, primary, both sexes (%)",SE.PRM.ENRR,99.821472,1990.0,113.699799,2017.0,13.878326,13.903147
9,Albania,ALB,"Gross enrolment ratio, secondary, both sexes (%)",SE.SEC.ENRR,90.261276,1990.0,95.765488,2017.0,5.504211,6.098087


In [5]:
# salvando a tabela de crescimento completa
df_crescimento.to_csv("data/processed/crescimento_por_pais_indicador.csv", index=False)

# RANKING: países que mais cresceram em matrícula secundária, por exemplo
indicador_ranking = "SE.SEC.ENRR"

ranking = df_crescimento[
    df_crescimento["Indicator Code"] == indicador_ranking
].sort_values("Crescimento_Percentual", ascending=False)

print(f"Top 10 países que mais cresceram em: {ranking['Indicator Name'].iloc[0]}")
ranking[["Country Name", "Valor_Inicial", "Valor_Final", "Crescimento_Percentual"]].head(10)

Top 10 países que mais cresceram em: Gross enrolment ratio, secondary, both sexes (%)


,Country Name,Valor_Inicial,Valor_Final,Crescimento_Percentual
183,Burundi,5.218590,42.480068,714.014321
723,Mali,6.775260,41.307480,509.681105
1131,Tanzania,5.344680,32.256081,503.517547
3,Afghanistan,11.222830,55.644409,395.814425
177,Burkina Faso,6.891890,33.666660,388.496770
321,Djibouti,10.247910,48.269562,371.018617
795,Mozambique,7.045640,32.431808,360.310327
1137,Thailand,28.504690,129.001709,352.563098
189,Cabo Verde,20.921329,92.898430,344.036933
891,Papua New Guinea,10.683060,40.346321,277.666345


In [6]:
def gerar_ranking(indicator_code, top_n=10, crescente=False):
    """
    Gera um ranking de países por crescimento em um indicador específico.
    
    indicator_code: código do indicador (ex: "SE.SEC.ENRR")
    top_n: quantos países mostrar
    crescente: se True, mostra os que MENOS cresceram (ou mais caíram)
    """
    dados = df_crescimento[df_crescimento["Indicator Code"] == indicator_code]
    dados = dados.sort_values("Crescimento_Percentual", ascending=crescente)
    
    nome_indicador = dados["Indicator Name"].iloc[0]
    print(f"Ranking: {nome_indicador}")
    print(f"({'Menores crescimentos' if crescente else 'Maiores crescimentos'})\n")
    
    return dados[["Country Name", "Valor_Inicial", "Valor_Final", 
                   "Crescimento_Percentual"]].head(top_n)

# testando a função com outro indicador: investimento em educação
gerar_ranking("SE.XPD.TOTL.GD.ZS", top_n=10)

Ranking: Government expenditure on education as % of GDP (%)
(Maiores crescimentos)



,Country Name,Valor_Inicial,Valor_Final,Crescimento_Percentual
1189,Tuvalu,7.55418,3.730834e+06,4.938757e+07
43,Argentina,1.06738,5.325490e+00,3.989310e+02
895,Paraguay,1.06267,4.955400e+00,3.663160e+02
1027,Solomon Islands,2.75382,1.000108e+01,2.631712e+02
523,Indonesia,0.99787,3.591810e+00,2.599477e+02
793,Mozambique,2.05194,6.483220e+00,2.159556e+02
1243,"Venezuela, RB",2.52841,6.877230e+00,1.719982e+02
331,Dominican Republic,0.78118,2.044570e+00,1.617284e+02
337,Ecuador,1.99764,4.963220e+00,1.484542e+02
979,Senegal,3.19778,7.396200e+00,1.312917e+02


In [7]:
# investigando o caso do Tuvalu antes de decidir o que fazer
print(df[(df["Country Name"] == "Tuvalu") & (df["Indicator Code"] == "SE.XPD.TOTL.GD.ZS")])

      Country Name Country Code  \
33068       Tuvalu          TUV   
33069       Tuvalu          TUV   
33070       Tuvalu          TUV   
33071       Tuvalu          TUV   
33072       Tuvalu          TUV   
33073       Tuvalu          TUV   
33074       Tuvalu          TUV   
33075       Tuvalu          TUV   
33076       Tuvalu          TUV   
33077       Tuvalu          TUV   
33078       Tuvalu          TUV   
33079       Tuvalu          TUV   
33080       Tuvalu          TUV   
33081       Tuvalu          TUV   
33082       Tuvalu          TUV   
33083       Tuvalu          TUV   
33084       Tuvalu          TUV   
33085       Tuvalu          TUV   
33086       Tuvalu          TUV   
33087       Tuvalu          TUV   
33088       Tuvalu          TUV   
33089       Tuvalu          TUV   
33090       Tuvalu          TUV   
33091       Tuvalu          TUV   
33092       Tuvalu          TUV   
33093       Tuvalu          TUV   
33094       Tuvalu          TUV   
33095       Tuvalu  

In [8]:
# ANTES do filtro: quantas linhas "impossíveis" existem no indicador de % do PIB?
suspeitos = df[(df["Indicator Code"] == "SE.XPD.TOTL.GD.ZS") & (df["Valor"] > 30)]
print(f"Linhas suspeitas (acima de 30% do PIB): {suspeitos.shape[0]}")
print(suspeitos["Country Name"].unique())

Linhas suspeitas (acima de 30% do PIB): 37
<StringArray>
['Tuvalu', 'Zimbabwe']
Length: 2, dtype: str


In [9]:
print(df[(df["Country Name"] == "Zimbabwe") & (df["Indicator Code"] == "SE.XPD.TOTL.GD.ZS")])

      Country Name Country Code  \
35924     Zimbabwe          ZWE   
35925     Zimbabwe          ZWE   
35926     Zimbabwe          ZWE   
35927     Zimbabwe          ZWE   
35928     Zimbabwe          ZWE   
35929     Zimbabwe          ZWE   
35930     Zimbabwe          ZWE   
35931     Zimbabwe          ZWE   
35932     Zimbabwe          ZWE   
35933     Zimbabwe          ZWE   
35934     Zimbabwe          ZWE   
35935     Zimbabwe          ZWE   
35936     Zimbabwe          ZWE   
35937     Zimbabwe          ZWE   
35938     Zimbabwe          ZWE   
35939     Zimbabwe          ZWE   
35940     Zimbabwe          ZWE   
35941     Zimbabwe          ZWE   
35942     Zimbabwe          ZWE   
35943     Zimbabwe          ZWE   
35944     Zimbabwe          ZWE   
35945     Zimbabwe          ZWE   
35946     Zimbabwe          ZWE   
35947     Zimbabwe          ZWE   
35948     Zimbabwe          ZWE   
35949     Zimbabwe          ZWE   
35950     Zimbabwe          ZWE   
35951     Zimbabwe  

In [10]:
# Regra de sanidade: % do PIB em educação não pode ultrapassar 100% (impossível fisicamente)
antes = df.shape[0]

condicao_erro = (df["Indicator Code"] == "SE.XPD.TOTL.GD.ZS") & (df["Valor"] > 100)
print("Linhas com erro de escala (>100% do PIB):", condicao_erro.sum())
print(df[condicao_erro][["Country Name", "Ano", "Valor"]])

# marcamos esses valores como NaN (dado inválido), em vez de simplesmente apagar a linha
df.loc[condicao_erro, "Valor"] = None

depois_validos = df["Valor"].notna().sum()
print(f"\nValores marcados como inválidos e tratados: {condicao_erro.sum()}")

Linhas com erro de escala (>100% do PIB): 21
      Country Name   Ano      Valor
33075       Tuvalu  1997  3730833.5
33076       Tuvalu  1998  3730833.5
33077       Tuvalu  1999  3730833.5
33078       Tuvalu  2000  3730833.5
33079       Tuvalu  2001  3730833.5
33080       Tuvalu  2002  3730833.5
33081       Tuvalu  2003  3730833.5
33082       Tuvalu  2004  3730833.5
33083       Tuvalu  2005  3730833.5
33084       Tuvalu  2006  3730833.5
33085       Tuvalu  2007  3730833.5
33086       Tuvalu  2008  3730833.5
33087       Tuvalu  2009  3730833.5
33088       Tuvalu  2010  3730833.5
33089       Tuvalu  2011  3730833.5
33090       Tuvalu  2012  3730833.5
33091       Tuvalu  2013  3730833.5
33092       Tuvalu  2014  3730833.5
33093       Tuvalu  2015  3730833.5
33094       Tuvalu  2016  3730833.5
33095       Tuvalu  2017  3730833.5

Valores marcados como inválidos e tratados: 21


In [11]:
df_ordenado = df.sort_values(["Country Code", "Indicator Code", "Ano"])

df_crescimento = df_ordenado.groupby(
    ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
).apply(calcular_crescimento, include_groups=False).reset_index()

# salvando a versão corrigida
df_crescimento.to_csv("data/processed/crescimento_por_pais_indicador.csv", index=False)

print("Shape:", df_crescimento.shape)

# testando de novo o ranking de investimento em educação
gerar_ranking("SE.XPD.TOTL.GD.ZS", top_n=10)

Shape: (1284, 10)
Ranking: Government expenditure on education as % of GDP (%)
(Maiores crescimentos)



,Country Name,Valor_Inicial,Valor_Final,Crescimento_Percentual
43,Argentina,1.06738,5.32549,398.931050
895,Paraguay,1.06267,4.95540,366.315980
1027,Solomon Islands,2.75382,10.00108,263.171150
523,Indonesia,0.99787,3.59181,259.947677
793,Mozambique,2.05194,6.48322,215.955643
1243,"Venezuela, RB",2.52841,6.87723,171.998223
331,Dominican Republic,0.78118,2.04457,161.728399
337,Ecuador,1.99764,4.96322,148.454180
979,Senegal,3.19778,7.39620,131.291722
751,Mexico,2.31177,5.31348,129.844664


In [12]:
def comparar_paises(lista_paises, indicator_code):
    """
    Compara um indicador específico entre uma lista de países.
    
    lista_paises: lista com nomes dos países (ex: ["Brazil", "Argentina"])
    indicator_code: código do indicador (ex: "SE.PRM.ENRR")
    """
    dados = df_crescimento[
        (df_crescimento["Country Name"].isin(lista_paises)) &
        (df_crescimento["Indicator Code"] == indicator_code)
    ]
    
    nome_indicador = dados["Indicator Name"].iloc[0] if not dados.empty else "N/A"
    print(f"Comparação: {nome_indicador}\n")
    
    return dados[["Country Name", "Valor_Inicial", "Valor_Final", 
                   "Crescimento_Percentual"]].sort_values("Crescimento_Percentual", ascending=False)

# testando: comparar Brasil com vizinhos da América do Sul
comparar_paises(["Brazil", "Argentina", "Chile", "Peru", "Colombia"], "SE.PRM.ENRR")

Comparação: Gross enrolment ratio, primary, both sexes (%)



,Country Name,Valor_Inicial,Valor_Final,Crescimento_Percentual
248,Colombia,101.686012,113.562721,11.679786
44,Argentina,106.561569,109.988892,3.216284
236,Chile,107.397011,101.643730,-5.357021
902,Peru,117.770607,101.695168,-13.649789
158,Brazil,165.645386,115.340790,-30.368848
